In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons
from scipy.stats import gaussian_kde
import plotly.express as px
import plotly.graph_objects as go
from torch.utils.data import Dataset


# Подготовим утилиты

Тут лежит код который:
датасет: набор 2д точек с классами
1) рисует датасет точек на плоскости
2) рисует датасет на этой плоскости и набор векторов на этой плоскости (по флагу нормировать длины и изображать длину цветом)
3) рисует график плотности вероятности по функции
4) рисует график плотности вероятности используя генератор семплов (нужно семплировать штуку для каждой точки несколько раз и рисовать график, промежуточные значения между сеткой можно интерполировать)

### Код

In [ ]:
def generate_2d_dataset(n_samples=500, n_classes=3, random_state=42):
    """
    Генерирует 2D датасет с классами.

    Parameters:
    -----------
    n_samples : int
        Количество точек для генерации
    n_classes : int
        Количество классов
    random_state : int
        Seed для воспроизводимости

    Returns:
    --------
    X : array-like, shape (n_samples, 2)
        Координаты точек
    y : array-like, shape (n_samples,)
        Метки классов
    """
    # Используем make_blobs для простых кластеров
    X, y = make_blobs(n_samples=n_samples, centers=n_classes, n_features=2,
                      random_state=random_state, cluster_std=1.0)
    return X, y

def generate_moons_dataset(n_samples=500, noise=0.1, random_state=42):
    """
    Генерирует нелинейный 2D датасет в форме двух полумесяцев.

    Parameters:
    -----------
    n_samples : int
        Количество точек для генерации
    noise : float
        Уровень шума
    random_state : int
        Seed для воспроизводимости

    Returns:
    --------
    X : array-like, shape (n_samples, 2)
        Координаты точек
    y : array-like, shape (n_samples,)
        Метки классов (2 класса)
    """
    X, y = make_moons(n_samples=n_samples, noise=noise, random_state=random_state)
    return X, y


In [ ]:
def plot_dataset(X, y, title="2D Dataset", figsize=(8, 6), ax=None):
    """
    Рисует датасет точек на плоскости с цветовым кодированием классов.

    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Координаты точек
    y : array-like, shape (n_samples,)
        Метки классов
    title : str
        Заголовок графика
    figsize : tuple
        Размер фигуры
    ax : matplotlib.axes.Axes or None
        Оси для рисования. Если None, создается новая фигура.
    """
    if ax is None:
        plt.figure(figsize=figsize)
        current_ax = plt.gca()
    else:
        current_ax = ax

    scatter = current_ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', alpha=0.7, s=50)
    if ax is None:
        plt.colorbar(scatter, label='Class')
    current_ax.set_xlabel('X1')
    current_ax.set_ylabel('X2')
    current_ax.set_title(title)
    current_ax.grid(True, alpha=0.3)
    if ax is None:
        plt.show()

def plot_vectors(X, y, vectors=None, normalize_lengths=False, title="Dataset with Vectors", figsize=(10, 8), ax=None, vector_func=None, grid_resolution=20):
    """
    Рисует датасет и векторы на плоскости.

    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Координаты точек
    y : array-like, shape (n_samples,)
        Метки классов
    vectors : array-like, shape (n_samples, 2) or None
        Векторы для каждой точки (по умолчанию None)
    normalize_lengths : bool
        Нормировать длины векторов и использовать цвет для отображения длины
    title : str
        Заголовок графика
    figsize : tuple
        Размер фигуры
    ax : matplotlib.axes.Axes or None
        Оси для рисования. Если None, создается новая фигура.
    vector_func : callable or None
        Функция для вычисления векторов на сетке (если vectors=None)
    grid_resolution : int
        Разрешение сетки для vector_func
    """
    if ax is None:
        fig, current_ax = plt.subplots(figsize=figsize)
    else:
        current_ax = ax
        fig = current_ax.figure

    # Рисуем датасет
    scatter = current_ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', alpha=0.7, s=50)
    current_ax.set_title(title)
    current_ax.set_xlabel('X1')
    current_ax.set_ylabel('X2')
    if ax is None:
        fig.colorbar(scatter, ax=current_ax, label='Class')
    current_ax.grid(True, alpha=0.3)

    if vectors is not None:
        if normalize_lengths:
            # Нормируем длины векторов и используем цвет для длины
            lengths = np.linalg.norm(vectors, axis=1)
            norm_vectors = vectors / (lengths[:, np.newaxis] + 1e-8)*(50/grid_resolution)/4
            colors = lengths  # Используем оригинальные длины для цвета

            for i in range(len(X)):
                current_ax.quiver(X[i, 0], X[i, 1], norm_vectors[i, 0], norm_vectors[i, 1],
                          colors[i], cmap='coolwarm', scale=5, width=0.005, alpha=0.8)
            sm = plt.cm.ScalarMappable(cmap='coolwarm', norm=plt.Normalize(vmin=lengths.min(), vmax=lengths.max()))
            sm.set_array([])
            if ax is None:
                fig.colorbar(sm, ax=current_ax, label='Vector Length')
        else:
            # Рисуем векторы как есть
            for i in range(len(X)):
                current_ax.quiver(X[i, 0], X[i, 1], vectors[i, 0], vectors[i, 1],
                          scale=20, width=0.002, alpha=0.6, color='red')
    elif vector_func is not None:
        # Создаем сетку для векторного поля
        x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
        y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_resolution),
                             np.linspace(y_min, y_max, grid_resolution))

        grid_points = np.vstack([xx.ravel(), yy.ravel()]).T
        grid_vectors = vector_func(grid_points)

        # Нормируем векторы для лучшей визуализации
        lengths = np.linalg.norm(grid_vectors, axis=1)
        norm_vectors = grid_vectors / (lengths[:, np.newaxis] + 1e-8)*(50/grid_resolution)/5

        # Рисуем векторное поле
        quiver = current_ax.quiver(xx, yy, norm_vectors[:, 0].reshape(xx.shape),
                  norm_vectors[:, 1].reshape(yy.shape),
                  lengths.reshape(xx.shape), cmap='coolwarm', scale=10, width=0.003+0.003*(1-grid_resolution/100))

        if ax is None:
            fig.colorbar(quiver, ax=current_ax, label='Vector Magnitude')

    if ax is None:
        plt.show()

In [ ]:
def plot_density_analytical(X, y=None, grid_size=100, title="2D Density Heatmap (Analytical)", figsize=(8, 6)):
    """
    Рисует 2D heatmap плотности вероятности по аналитической функции (используя KDE).

    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Координаты точек
    y : array-like, shape (n_samples,) or None
        Метки классов (если None, общая плотность)
    grid_size : int
        Размер сетки для вычисления плотности
    title : str
        Заголовок графика
    figsize : tuple
        Размер фигуры
    """
    # Создаем сетку
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_size),
                         np.linspace(y_min, y_max, grid_size))

    if y is None:
        # Общая плотность
        kde = gaussian_kde(X.T)
        Z = np.reshape(kde(np.vstack([xx.ravel(), yy.ravel()])), xx.shape)

        plt.figure(figsize=figsize)
        sns.heatmap(Z, xticklabels=False, yticklabels=False, cmap='viridis', cbar_kws={'label': 'Density'})
        plt.xlabel('X1')
        plt.ylabel('X2')
        plt.title(title)
        plt.show()
    else:
        # Плотность по классам
        n_classes = len(np.unique(y))
        fig, axes = plt.subplots(1, n_classes, figsize=(4*n_classes, 4))
        if n_classes == 1:
            axes = [axes]

        for i in range(n_classes):
            mask = y == i
            if np.sum(mask) > 0:
                kde_class = gaussian_kde(X[mask].T)
                Z_class = np.reshape(kde_class(np.vstack([xx.ravel(), yy.ravel()])), xx.shape)

                sns.heatmap(Z_class, xticklabels=False, yticklabels=False,
                           cmap='viridis', ax=axes[i], cbar_kws={'label': f'Density Class {i}'})
                axes[i].set_title(f'Class {i}')
                axes[i].set_xlabel('X1')
                axes[i].set_ylabel('X2')
            else:
                axes[i].set_title(f'Class {i} (Empty)')

        plt.tight_layout()
        plt.show()

def plot_density_plotly(X, y=None, grid_size=50, title="Interactive 2D Density Heatmap"):
    """
    Рисует интерактивный 2D heatmap плотности вероятности с помощью Plotly.

    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Координаты точек
    y : array-like, shape (n_samples,) or None
        Метки классов (если None, общая плотность)
    grid_size : int
        Размер сетки для вычисления плотности
    title : str
        Заголовок графика
    """
    # Создаем сетку
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_size),
                         np.linspace(y_min, y_max, grid_size))

    if y is None:
        # Общая плотность
        kde = gaussian_kde(X.T)
        Z = np.reshape(kde(np.vstack([xx.ravel(), yy.ravel()])), xx.shape)

        fig = px.imshow(Z, x=np.linspace(x_min, x_max, grid_size),
                        y=np.linspace(y_max, y_min, grid_size),  # y reversed for correct orientation
                        color_continuous_scale='Viridis',
                        title=title,
                        labels={'color': 'Density', 'x': 'X1', 'y': 'X2'})
        fig.show()
    else:
        # Плотность по классам (показываем в отдельных фигурах)
        n_classes = len(np.unique(y))
        for i in range(n_classes):
            mask = y == i
            if np.sum(mask) > 0:
                kde_class = gaussian_kde(X[mask].T)
                Z_class = np.reshape(kde_class(np.vstack([xx.ravel(), yy.ravel()])), xx.shape)

                fig = px.imshow(Z_class, x=np.linspace(x_min, x_max, grid_size),
                               y=np.linspace(y_max, y_min, grid_size),
                               color_continuous_scale='Viridis',
                               title=f'{title} - Class {i}',
                               labels={'color': f'Density Class {i}', 'x': 'X1', 'y': 'X2'})
                fig.show()


In [ ]:
def plot_density_samples(X, n_samples_per_point=100, grid_resolution=50, title="2D Density from Samples", figsize=(8, 6)):
    """
    Рисует 2D heatmap плотности вероятности, используя генерацию семплов.
    Для каждой точки сетки семплируем несколько значений и интерполируем.

    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Исходные точки для оценки плотности
    n_samples_per_point : int
        Количество семплов для каждой точки сетки
    grid_resolution : int
        Разрешение сетки
    title : str
        Заголовок графика
    figsize : tuple
        Размер фигуры
    """
    # Создаем сетку точек для оценки
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_resolution),
                         np.linspace(y_min, y_max, grid_resolution))
    grid_points = np.vstack([xx.ravel(), yy.ravel()]).T

    # Функция для семплирования вокруг точки (простая: добавляем шум)
    def sample_around_point(point, n_samples, bandwidth=0.5):
        noise = np.random.normal(0, bandwidth, (n_samples, 2))
        return point + noise

    # Для каждой точки сетки генерируем семплы и оцениваем локальную плотность
    # Здесь упрощенная версия: используем расстояния до исходных точек
    from scipy.spatial.distance import cdist

    densities = []
    for point in grid_points:
        # Семплируем вокруг точки
        samples = sample_around_point(point, n_samples_per_point)

        # Вычисляем среднюю "принадлежность" семплов к датасету (по расстоянию)
        distances = cdist(samples, X)
        # Плотность обратно пропорциональна расстоянию (гауссово ядро)
        kernel_density = np.mean(np.exp(-distances**2 / (2 * 0.5**2)))
        densities.append(kernel_density)

    Z = np.array(densities).reshape(xx.shape)

    plt.figure(figsize=figsize)
    sns.heatmap(Z, xticklabels=False, yticklabels=False, cmap='viridis',
                cbar_kws={'label': 'Estimated Density'})
    plt.xlabel('X1')
    plt.ylabel('X2')
    plt.title(title)
    plt.show()


# DDPM + DDIM (utils)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F

In [ ]:
class LinearScheduler:
    def __init__(self, beta_start=0.0001, beta_end=0.02, num_steps=1000):
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.num_steps = num_steps

    def get_betas(self):
        return torch.linspace(self.beta_start, self.beta_end, self.num_steps)

class CosineScheduler:
    def __init__(self, beta_start=0.0001, beta_end=0.02, num_steps=1000):
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.num_steps = num_steps

    def get_betas(self):
        alphas_cumprod = torch.cos(torch.linspace(0, torch.pi, self.num_steps)) ** 2
        return 1 - alphas_cumprod / alphas_cumprod[0]

class NoiseSchedule(nn.Module):
    def __init__(self, scheduler_type='linear', beta_start=0.0001, beta_end=0.04, num_steps=1000):
        super().__init__() # Corrected super().__init__() call
        self.scheduler_type = scheduler_type
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.num_steps = num_steps

        self._betas = None
        #self._alphas = None
        #self._alphas_cumprod = None
        #self._sqrt_alphas_cumprod = None
        #self._sqrt_one_minus_alphas_cumprod = None
        self._setup()

    def _setup(self):
        if self.scheduler_type == 'linear':
            self._betas = LinearScheduler(self.beta_start, self.beta_end, self.num_steps).get_betas()
        elif self.scheduler_type == 'cosine':
            self._betas = CosineScheduler(self.beta_start, self.beta_end, self.num_steps).get_betas()
        else:
            raise ValueError(f"Unsupported scheduler type: {self.scheduler_type}")

        _alphas = 1.0 - self._betas
        self.register_buffer('_alphas', _alphas)
        _alphas_cumprod = torch.cumprod(self._alphas, dim=0)
        self.register_buffer('_alphas_cumprod', _alphas_cumprod)
        _sqrt_alphas_cumprod = torch.sqrt(self._alphas_cumprod)
        self.register_buffer('_sqrt_alphas_cumprod', _sqrt_alphas_cumprod)
        _sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self._alphas_cumprod)
        self.register_buffer('_sqrt_one_minus_alphas_cumprod', _sqrt_one_minus_alphas_cumprod)

    @property
    def betas(self):
        return self._betas

    @property
    def alphas(self):
        return self._alphas

    @property
    def alphas_cumprod(self):
        return self._alphas_cumprod

    @property
    def sqrt_alphas_cumprod(self):
        return self._sqrt_alphas_cumprod

    @property
    def sqrt_one_minus_alphas_cumprod(self):
        return self._sqrt_one_minus_alphas_cumprod

    def add_noise(self, x, t):
        """Forward diffusion process (training)"""
        if torch.cuda.is_available():
            with torch.amp.autocast('cuda', torch.float32):
                noise = torch.randn_like(x)
                sqrt_alpha_bar = self.sqrt_alphas_cumprod[t][:, None]
                sqrt_one_minus_alpha_bar = self.sqrt_one_minus_alphas_cumprod[t][:, None]
                return sqrt_alpha_bar * x + sqrt_one_minus_alpha_bar * noise, noise

        noise = torch.randn_like(x)
        sqrt_alpha_bar = self.sqrt_alphas_cumprod[t][:, None]
        sqrt_one_minus_alpha_bar = self.sqrt_one_minus_alphas_cumprod[t][:, None]
        return sqrt_alpha_bar * x + sqrt_one_minus_alpha_bar * noise, noise


    def denoise_ddpm(self, x, pred_noise, t):
        """DDPM reverse process"""
        sqrt_alpha_bar = self.sqrt_alphas_cumprod[t][:, None]
        sqrt_one_minus_alpha_bar = self.sqrt_one_minus_alphas_cumprod[t][:, None]
        return (x - sqrt_one_minus_alpha_bar * pred_noise) / sqrt_alpha_bar

    def denoise_ddim(self, x, pred_noise, t, eta=0.0):
        """DDIM reverse process"""
        alpha_bar = self.alphas_cumprod[t]
        alpha_bar_prev = self.alphas_cumprod[t-1] if t[0] > 0 else torch.zeros_like(alpha_bar)

        sigma = eta * torch.sqrt((1 - alpha_bar_prev) / (1 - alpha_bar) * (1 - alpha_bar / alpha_bar_prev))
        sqrt_alpha_bar = torch.sqrt(alpha_bar)[:, None]
        sqrt_alpha_bar_prev = torch.sqrt(alpha_bar_prev)[:, None]

        pred_x0 = (x - torch.sqrt(1 - alpha_bar)[:, None] * pred_noise) / sqrt_alpha_bar
        dir_xt = torch.sqrt(1 - alpha_bar_prev - sigma**2)[:, None] * pred_noise
        noise = sigma[:, None] * torch.randn_like(x)

        return sqrt_alpha_bar_prev * pred_x0 + dir_xt + noise

    def sample_timesteps(self, n_samples, device):
        """Sample random timesteps for training"""
        return torch.randint(0, self.num_steps, (n_samples,), device=device)

    def sample_ddim_timesteps(self, n_samples, n_steps=None, device='cuda'):
        """Sample timesteps for DDIM sampling (evenly spaced)"""
        if n_steps is None:
            n_steps = self.num_steps
        step = self.num_steps // n_steps
        return torch.arange(0, self.num_steps, step, device=device)[:n_samples]

    def get_init_noise(self, shape, device):
        """Get initial noise for sampling"""
        return torch.randn(shape, device=device)

# Конструирование модели (MLP + Attention)

In [ ]:


from scipy.stats import norm


class MLP(nn.Module):
    def __init__(self, in_features, hidden_features, out_features, act_layer=nn.GELU):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.act = act_layer()

    def forward(self, x):
        x = self.act(self.fc1(x))
        return self.fc2(x)

class Attention(nn.Module):
    def __init__(self, dim, num_heads=4, qkv_bias=False, attn_drop=0.0, proj_drop=0.0):
        super(Attention, self).__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class CrossAttention(nn.Module):
    def __init__(self, dim, dim_x, dim_y, num_heads=4, qkv_bias=False, attn_drop=0.0, proj_drop=0.0):
        super(CrossAttention, self).__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))
        # Separate projections for Q (from x) and KV (from y)
        self.q_proj = nn.Linear(dim_x, dim, bias=qkv_bias)
        self.kv_proj = nn.Linear(dim_y, dim * 2, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, y):
        B, N, C = x.shape
        M = y.shape[1]

        # Q from x: [B, N, C] -> [B, num_heads, N, head_dim]
        q = self.q_proj(x).reshape(B, N, self.num_heads, C // self.num_heads).permute(0, 2, 1, 3)

        # K, V from y: [B, M, C] -> [B, num_heads, M, head_dim]
        kv = self.kv_proj(y).reshape(B, M, 2, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]

        # Apply temperature to queries
        q = q * self.temperature

        # Attention: Q @ K^T
        attn = (q @ k.transpose(-2, -1)) * self.scale  # [B, num_heads, N, M]
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # Apply attention to values: [B, num_heads, N, M] @ [B, num_heads, M, head_dim] -> [B, num_heads, N, head_dim]
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)  # [B, N, C]
        out = self.proj(out)
        out = self.proj_drop(out)

        return out

class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=False,
                 drop=0.0, attn_drop=0.0, act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super(Block, self).__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(in_features=dim, hidden_features=mlp_hidden_dim, out_features=dim, act_layer=act_layer)


    def forward(self, x):
        x = x + (self.attn(self.norm1(x)))
        x = x + (self.mlp(self.norm2(x)))
        return x

class CrossBlock(nn.Module):
    def __init__(self, dim, dim_x, dim_y, num_heads, mlp_ratio=4.0, qkv_bias=False,
                 drop=0.0, attn_drop=0.0, act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super(CrossBlock, self).__init__()
        self.norm1_x = norm_layer(dim_x)
        self.norm1_y = norm_layer(dim_y)
        self.cross_attn = CrossAttention(dim, dim_x, dim_y, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = norm_layer(dim)
        #self.y_mlp = MLP(in_features=dim_y, hidden_features=dim, out_features=dim_y)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(in_features=dim, hidden_features=mlp_hidden_dim, out_features=dim, act_layer=act_layer)
    def forward(self, x, y):
        #y = y + self.y_mlp(y)
        x = x + (self.cross_attn(self.norm1_x(x), self.norm1_y(y)))
        x = x + (self.mlp(self.norm2(x)))
        return x

class RoPE(nn.Module):
    def __init__(self, dim, device='cuda' if torch.cuda.is_available() else 'cpu'):
        super(RoPE, self).__init__()
        self.base = 10000
        self.dim = dim
        inv_freq = 1.0 / (self.base ** (torch.arange(0, dim, 2, device=device).float() / dim))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, seq_len, device='cuda' if torch.cuda.is_available() else 'cpu'):
        """
        Generate rotary position embeddings for sequence length.
        Returns: [seq_len, dim]
        """
        # Generate position indices
        positions = torch.arange(seq_len, device=device, dtype=torch.float32)
        # Compute frequencies: [seq_len, dim//2]
        freqs = torch.outer(positions, self.inv_freq.to(device))
        # Create sin and cos embeddings: [seq_len, dim//2] each
        sin = torch.sin(freqs)
        cos = torch.cos(freqs)
        # Interleave sin and cos to get [seq_len, dim]
        emb = torch.cat([sin, cos], dim=-1)
        return emb

class Embedding(nn.Module):
    def __init__(self, dim, device='cuda' if torch.cuda.is_available() else 'cpu', num_embeddings=10000):
        super(Embedding, self).__init__()
        self.dim = dim
        self.device = device
        self.embedding = nn.Embedding(num_embeddings, dim)
    def forward(self, x):
        return self.embedding(x)

class Modulation(nn.Module):
    def __init__(self, dim, device='cuda' if torch.cuda.is_available() else 'cpu'):
        super(Modulation, self).__init__()
        self.dim = dim
        self.device = device
        self.modulation = nn.Linear(dim, dim)
    def forward(self, x):
        return self.modulation(x)

# class that take x and timestamp embedding and return x with timestamp
class TimeMLP(nn.Module):
    def __init__(self, dim, device='cuda' if torch.cuda.is_available() else 'cpu'):
        super(TimeMLP, self).__init__()
        self.dim = dim
        self.device = device
        self.mlp = MLP(dim, dim, dim)
    def forward(self, x, timestamp):
        timestamp_emb = self.timestamp_embedding(timestamp)
        return self.mlp(x) + timestamp_emb



class DiT(nn.Module):
    def __init__(self, dim, num_heads, condition_dim, timestamp_dim, num_layers, act_layer=nn.GELU, norm_layer=nn.LayerNorm, device='cuda' if torch.cuda.is_available() else 'cpu'):
        super(DiT, self).__init__()
        self.dim = dim
        self.device = device
        self.rope = RoPE(dim, device)
        # add condition and timestamp cross attention blocks
        self.condition_embedding = nn.Linear(condition_dim, dim)
        self.timestamp_embedding = nn.Linear(timestamp_dim, dim)

        self.time_cross_attn_blocks = nn.ModuleList([
            CrossBlock(dim, dim, dim, num_heads, mlp_ratio=4.0, qkv_bias=False,
                      drop=0.0, attn_drop=0.0, act_layer=act_layer, norm_layer=norm_layer)
            for _ in range(num_layers)
        ])
        self.cross_attn_blocks = nn.ModuleList([
            CrossBlock(dim, dim, dim, num_heads, mlp_ratio=4.0, qkv_bias=False,
                      drop=0.0, attn_drop=0.0, act_layer=act_layer, norm_layer=norm_layer)
            for _ in range(num_layers)
        ])
        self.blocks = nn.ModuleList([
            Block(dim, num_heads, mlp_ratio=4.0, qkv_bias=False,
                  drop=0.0, attn_drop=0.0, act_layer=act_layer, norm_layer=norm_layer)
            for _ in range(num_layers)
        ])
        self.norm1 = norm_layer(dim)
        self.norm2 = norm_layer(dim)

    def forward(self, x, condition, timestamp):
        """
        Handle different input dimensions:
        - [T] -> single sequence of T features (treat as [1, 1, T])
        - [B, T] -> batch of sequences, T is feature dimension (e.g., 2D points where T=dim)
        - [B, C, T] -> batch with C channels, T sequence length
        - [B, C, H, W] -> spatial data with C channels, flatten H*W as sequence

        Internal format: [B, seq_len, feature_dim]
        Output: same shape as input
        """
        original_shape = x.shape
        B, seq_len, input_dim = None, None, None

        if len(x.shape) == 1:
            # [T] -> [1, 1, T] where T is feature dimension
            T = x.shape[0]
            x = x.unsqueeze(0).unsqueeze(0)  # [1, 1, T]
            B, seq_len, input_dim = 1, 1, T
        elif len(x.shape) == 2:
            # [B, T] -> [B, 1, T] where T is feature dimension (e.g., T=2 for 2D points)
            # This is the main case for our 2D point dataset
            B, T = x.shape
            x = x.unsqueeze(1)  # [B, 1, T]
            seq_len, input_dim = 1, T
        elif len(x.shape) == 3:
            # [B, C, T] -> [B, T, C] where T is sequence length, C is feature dimension
            B, C, T = x.shape
            x = x.permute(0, 2, 1)  # [B, T, C]
            seq_len, input_dim = T, C
        elif len(x.shape) == 4:
            # [B, C, H, W] -> [B, H*W, C] where C is feature dimension, H*W is sequence
            B, C, H, W = x.shape
            x = x.reshape(B, C, H * W).permute(0, 2, 1)  # [B, H*W, C]
            seq_len, input_dim = H * W, C
        else:
            raise ValueError(f"Unsupported input shape: {original_shape}")

        # Project input features to model dimension if needed
        # input_dim should match self.dim for proper processing
        if input_dim != self.dim:
            if not hasattr(self, 'input_proj'):
                self.input_proj = nn.Linear(input_dim, self.dim).to(x.device)
            x = self.input_proj(x)

        # Generate positional embeddings for sequence length: [seq_len, dim]
        pos_emb = self.rope(seq_len, device=x.device)  # [seq_len, dim]
        pos_emb = pos_emb.unsqueeze(0)  # [1, seq_len, dim] for broadcasting

        # Ensure condition and timestamp embeddings have proper shape [B, 1, dim]
        timestamp_ = self.timestamp_embedding(timestamp)  # [B, timestamp_dim] or [B, dim]
        condition_ = self.condition_embedding(condition)  # [B, condition_dim] or [B, dim]

        if len(timestamp_.shape) == 2:
            timestamp_ = timestamp_.unsqueeze(1)  # [B, 1, dim]
        if len(condition_.shape) == 2:
            condition_ = condition_.unsqueeze(1)  # [B, 1, dim]

        x = self.norm1(x)

        for i in range(len(self.time_cross_attn_blocks)):
            x = x + pos_emb
            x = self.time_cross_attn_blocks[i](x, timestamp_)
            x = x + pos_emb
            x = self.cross_attn_blocks[i](x, condition_)
            x = x + pos_emb
            x = self.blocks[i](x)

        x = self.norm2(x)

        # Project back to original feature dimension and reshape to original shape
        if input_dim != self.dim:
            if not hasattr(self, 'output_proj'):
                self.output_proj = nn.Linear(self.dim, input_dim).to(x.device)
            x = self.output_proj(x)

        # Reshape back to original input shape
        if len(original_shape) == 1:
            # [1, 1, T] -> [T]
            x = x.squeeze(0).squeeze(0)
        elif len(original_shape) == 2:
            # [B, 1, T] -> [B, T]
            x = x.squeeze(1)
        elif len(original_shape) == 3:
            # [B, T, C] -> [B, C, T]
            x = x.permute(0, 2, 1)
        elif len(original_shape) == 4:
            # [B, H*W, C] -> [B, C, H, W]
            B, C, H, W = original_shape
            x = x.permute(0, 2, 1).reshape(B, C, H, W)

        return x


## **Пайплайн**

In [ ]:
GL_BETA_START = 1e-4
GL_BETA_END = 0.02
# new aproxed
GL_BETA_START = 1e-5
GL_BETA_END = 1e-3

class Pipeline(nn.Module):
    def __init__(self, dim, num_heads, condition_dim, timestamp_dim, num_layers, scheduler_type='cosine', max_condition=10000, max_timestamp=1000, act_layer=nn.GELU, norm_layer=nn.LayerNorm, device='cuda' if torch.cuda.is_available() else 'cpu'):
        super(Pipeline, self).__init__()
        self.dim = dim
        self.device = device

        self.dit = DiT(dim, num_heads, condition_dim, timestamp_dim, num_layers, act_layer, norm_layer, device)
        # Explicitly add noise_schedule as a module

        noise_schedule_instance = NoiseSchedule(scheduler_type=scheduler_type, beta_start=GL_BETA_START, beta_end=GL_BETA_END, num_steps=max_timestamp)
        self.add_module('noise_schedule', noise_schedule_instance)
        self.noise_schedule = noise_schedule_instance


        self.condition_embedder = Embedding(condition_dim, device, num_embeddings=max_condition+1)
        self.timestamp_embedder = Embedding(timestamp_dim, device, num_embeddings=max_timestamp+1)
        self.unk_cond_id = max_condition
        self.und_timestamp_id = max_timestamp


    def forward(self, x, condition=None, timestamp=None):
        if condition is not None:
            condition = self.condition_embedder(condition)
        else:
            condition = self.condition_embedder(self.unk_cond_id)
        if timestamp is not None:
            timestamp = self.timestamp_embedder(timestamp)
        else:
            timestamp = self.timestamp_embedder(self.und_timestamp_id)

        return self.dit(x, condition, timestamp)

    def loss(self, pred, target):
        return F.mse_loss(pred, target, reduction='none')

    def sample(self, n_samples, condition=None, timestamp=None):
        x = self.noise_schedule.get_init_noise((n_samples, self.dim), self.device)
        for i in range(self.noise_schedule.num_steps):
            x = self(x, condition, timestamp)
        return x

    def get_timesteps(self, n_samples):
        return self.noise_schedule.sample_timesteps(n_samples, self.device)

    def get_timestep_embedding(self, t):
        return self.timestamp_embedder(t)

    def get_condition_embedding(self, condition):
        return self.condition_embedder(condition)

    def get_init_noise(self, shape):
        return self.noise_schedule.get_init_noise(shape, self.device)

# Данные и датасет

In [ ]:
def load_pipeline(dim, num_heads, condition_dim, timestamp_dim, max_condition, max_timestamp, num_layers, act_layer, norm_layer, device):
    pipeline = Pipeline(dim, num_heads, condition_dim, timestamp_dim, max_condition, max_timestamp, num_layers, act_layer, norm_layer, device)
    return pipeline

def prepare_train_pipeline(pipeline, dataloader, num_epochs, lr, device):
    pipeline.to(device)
    optimizer = torch.optim.AdamW(pipeline.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_epochs+1)
    return pipeline, optimizer, scheduler

def visualize_training_progress_fast(pipeline, X_sample, y_sample, device, epoch,
                                     grid_resolution=40, figsize=(16, 4)):
    """
    Быстрая визуализация прогресса обучения (на торче, минимальные вычисления).

    Parameters:
    -----------
    pipeline : Pipeline
        Модель
    X_sample : torch.Tensor
        Небольшой набор точек для визуализации
    y_sample : torch.Tensor
        Метки классов
    device : str
        Устройство
    epoch : int
        Номер эпохи
    grid_resolution : int
        Разрешение сетки (маленькое для скорости)
    figsize : tuple
        Размер фигуры
    """
    pipeline.eval()

    mts = pipeline.und_timestamp_id
    with torch.no_grad():
        # Параметры для визуализации: unconditional, разные таймстемпы
        configs = [
            (None, int(mts*0.08), f"Unconditional, t={int(mts*0.08)}"),
            (None, int(mts*0.16), f"Unconditional, t={int(mts*0.16)}"),
            (None, int(mts*0.32), f"Unconditional, t={int(mts*0.32)}"),
            (None, int(mts*0.64), f"Unconditional, t={int(mts*0.64)}")
        ]
        # Создаем 3 графика: unconditional и несколько таймстемпов
        fig, axes = plt.subplots(1, len(configs), figsize=figsize)

        # Вычисляем границы датасета
        if X_sample[:, 0].min().item()<0:
            x_min, x_max = X_sample[:, 0].min().item() * 1.2, X_sample[:, 0].max().item() * 1.2
        else:
            x_min, x_max = X_sample[:, 0].min().item() * 0.8, X_sample[:, 0].max().item() * 1.2
        if X_sample[:, 1].min().item()<0:
            y_min, y_max = X_sample[:, 1].min().item() * 1.2, X_sample[:, 1].max().item() * 1.2
        else:
            y_min, y_max = X_sample[:, 1].min().item() * 0.8, X_sample[:, 1].max().item() * 1.2
        # Создаем сетку на торче
        xx = torch.linspace(x_min, x_max, grid_resolution, device=device)
        yy = torch.linspace(y_min, y_max, grid_resolution, device=device)
        xx_grid, yy_grid = torch.meshgrid(xx, yy, indexing='xy')
        grid_points = torch.stack([xx_grid.flatten(), yy_grid.flatten()], dim=1)



        for idx, (cond, timestamp, title) in enumerate(configs):
            # Подготовка condition - используем unk_cond_id для unconditional
            condition_tensor = torch.full((grid_points.shape[0],), pipeline.unk_cond_id,
                                         dtype=torch.long, device=device)

            # Таймстемп
            t_tensor = torch.full((grid_points.shape[0],), timestamp,
                                 dtype=torch.long, device=device)

            # ИСПРАВЛЕНИЕ: НЕ добавляем шум! Считаем grid_points уже зашумленными точками
            # Получаем предсказание модели (предсказывает шум)
            pred_noise = pipeline(grid_points, condition=condition_tensor, timestamp=t_tensor)

            # Вычисляем направление денойзинга (к чистым данным)
            sqrt_alpha_bar = pipeline.noise_schedule.sqrt_alphas_cumprod[timestamp]
            sqrt_one_minus_alpha_bar = pipeline.noise_schedule.sqrt_one_minus_alphas_cumprod[timestamp]

            # Предсказываем x_0 (чистые данные)
            pred_x0 = (grid_points - sqrt_one_minus_alpha_bar * pred_noise) / sqrt_alpha_bar

            # Направление денойзинга
            denoising_direction = pred_x0 - grid_points

            # Вычисляем длины векторов
            lengths = torch.norm(denoising_direction, dim=1).cpu().numpy()

            # Нормируем для визуализации
            denoising_direction_norm = denoising_direction / (torch.norm(denoising_direction, dim=1, keepdim=True) + 1e-8)

            # Reshape для визуализации
            u = denoising_direction_norm[:, 0].reshape(grid_resolution, grid_resolution).cpu().numpy()
            v = denoising_direction_norm[:, 1].reshape(grid_resolution, grid_resolution).cpu().numpy()
            lengths_grid = lengths.reshape(grid_resolution, grid_resolution)

            #x_dots = torch.zeros_like(X_sample)
            #for i, el in enumerate(X_sample):
            x_dots, _ = pipeline.noise_schedule.add_noise(X_sample, torch.full((X_sample.shape[0],), timestamp,
                                 dtype=torch.long, device=device))
                #x_dots[i] = x_dot

            # Отображаем точки датасета
            axes[idx].scatter(x_dots[:, 0].cpu().numpy(),
                            x_dots[:, 1].cpu().numpy(),
                            c=y_sample.cpu().numpy(), cmap='tab10',
                            alpha=0.6, s=10, zorder=1)

            # Векторное поле
            xx_np = xx_grid.cpu().numpy()
            yy_np = yy_grid.cpu().numpy()
            u = u * 0.5
            v = u * 0.5
            axes[idx].quiver(xx_np, yy_np, u, v, lengths_grid,
                           cmap='coolwarm', scale=10, width=0.003, alpha=0.7, zorder=2, lw=4)

            axes[idx].set_title(f"{title}\nEpoch {epoch}")
            axes[idx].set_xlabel('X1')
            axes[idx].set_ylabel('X2')
            axes[idx].grid(True, alpha=0.3)
            axes[idx].set_xlim(x_min, x_max)
            axes[idx].set_ylim(y_min, y_max)

        plt.tight_layout()
        plt.show()

    pipeline.train()
import torch.amp
def train_pipeline(pipeline, optimizer, scheduler, dataloader, num_epochs, device,
                  uncond_prob=0.1, visualize_every=10, vis_sample_size=200):
    """
    Обучение pipeline с поддержкой unconditional training и визуализацией прогресса.

    Parameters:
    -----------
    pipeline : Pipeline
        Модель
    optimizer : torch.optim.Optimizer
        Оптимизатор
    scheduler : torch.optim.lr_scheduler
        Планировщик learning rate
    dataloader : DataLoader
        Данные для обучения
    num_epochs : int
        Количество эпох
    device : str
        Устройство
    uncond_prob : float
        Вероятность unconditional training (condition заменяется на unk_cond_id)
    visualize_every : int
        Визуализировать каждые N эпох (0 = не визуализировать)
    vis_sample_size : int
        Размер выборки для визуализации
    """
    pipeline.to(device)
    pipeline.train()

    # Подготовка данных для визуализации
    if visualize_every > 0:
        X_vis_list = []
        y_vis_list = []
        for x_batch, y_batch in dataloader:
            X_vis_list.append(x_batch)
            y_vis_list.append(y_batch)
            if sum(len(x) for x in X_vis_list) >= vis_sample_size:
                break
        X_vis = torch.cat(X_vis_list, dim=0)[:vis_sample_size].to(device)
        y_vis = torch.cat(y_vis_list, dim=0)[:vis_sample_size].to(device)

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0

        for x, condition in dataloader:
            optimizer.zero_grad()
            if torch.cuda.is_available():
                with torch.amp.autocast("cuda", torch.float32):
                    x = x.to(device)
                    condition = condition.to(device)
                    batch_size = x.shape[0]
                    # Семплируем таймстемпы
                    t = pipeline.noise_schedule.sample_timesteps(batch_size, device)

                    # Unconditional training: рандомно заменяем condition на unk_cond_id
                    if uncond_prob > 0:
                        uncond_mask = torch.rand(batch_size, device=device) < uncond_prob
                        if uncond_mask.any():
                            condition = condition.clone()
                            condition[uncond_mask] = pipeline.unk_cond_id

                    # Добавляем шум
                    x_noised, noise = pipeline.noise_schedule.add_noise(x, t)

                    # Предсказание
                    pred = pipeline(x_noised, condition, t)

                    # Loss
                    loss = pipeline.loss(pred, noise)
            else:
                x = x.to(device)
                condition = condition.to(device)
                batch_size = x.shape[0]
                # Семплируем таймстемпы
                t = pipeline.noise_schedule.sample_timesteps(batch_size, device)

                # Unconditional training: рандомно заменяем condition на unk_cond_id
                if uncond_prob > 0:
                    uncond_mask = torch.rand(batch_size, device=device) < uncond_prob
                    if uncond_mask.any():
                        condition = condition.clone()
                        condition[uncond_mask] = pipeline.unk_cond_id

                # Добавляем шум
                x_noised, noise = pipeline.noise_schedule.add_noise(x, t)

                # Предсказание
                pred = pipeline(x_noised, condition, t)

                # Loss
                loss = pipeline.loss(pred, noise)
            """
            if epoch<4:
                t = t.view(-1, 1)
                loss = loss*t
            if epoch>10 and epoch<18:
                t = t.view(-1, 1)
                loss = loss*(1/(t+torch.ones_like(t)))
            """
            loss = loss.mean()

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1

        scheduler.step()

        avg_loss = epoch_loss / num_batches

        # Вывод прогресса
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}, LR: {scheduler.get_last_lr()[0]:.6f}")

        # Визуализация прогресса
        if visualize_every > 0 and (epoch + 1) % visualize_every == 0:
            visualize_training_progress_fast(pipeline, X_vis, y_vis, device, epoch + 1)

def prepare_eval_pipeline(pipeline, dataloader, device):
    pipeline.to(device)
    return pipeline

def eval_pipeline(pipeline, dataloader, device):
    pipeline.to(device)
    pipeline.eval()
    samples = []
    with torch.no_grad():
        for x, condition in dataloader:
            x = x.to(device)
            condition = condition.to(device)
            # ddim sampling
            x = pipeline.sample(x.shape[0], condition)
            # save image
            samples.append(x)
    return torch.cat(samples, dim=0)


In [ ]:
# Сконструируем датасет в форме вращающихся лучей  , каждый луч -- свой класс
def construct_dataset(n_samples, n_classes):
    # Создаем n_samples точек для каждого класса
    X = np.zeros((n_samples * n_classes, 2))
    y = np.zeros(n_samples * n_classes, dtype=int)
    for i in range(n_classes):

        theta = np.linspace(0, 1.6 * np.pi / (n_classes*0.9), n_samples) + np.pi*2*(i/n_classes)+np.pi/2
        r = np.sqrt(np.linspace(0, 100, n_samples))  # Радиус
        X[i*n_samples:(i+1)*n_samples, 0] = np.cos(theta)*r+np.random.normal(0, r/120)
        X[i*n_samples:(i+1)*n_samples, 0] = X[i*n_samples:(i+1)*n_samples, 0] / 11
        X[i*n_samples:(i+1)*n_samples, 1] = np.sin(theta)*r+np.random.normal(0, r/120)
        X[i*n_samples:(i+1)*n_samples, 1] = X[i*n_samples:(i+1)*n_samples, 1] / 11
        y[i*n_samples:(i+1)*n_samples] = i
    return X, y

# Сконструируем датасет в форме двух полумесяцев
def construct_moons_dataset(n_samples, noise):
    X, y = make_moons(n_samples=n_samples, noise=noise)
    return X, y


In [ ]:
# Торчевый датасет и даталоадер
from torch.utils.data import DataLoader

class TorchDataset(Dataset):
    def __init__(self, X, y):
        """
        X: numpy array of shape (n_samples, 2)
        y: numpy array of shape (n_samples,) with class labels
        """
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def create_dataset_loader(construct_func, n_samples, batch_size, num_workers=0, noise=None, **kwargs):
    """Wrapper to create dataset and dataloader from constructor function"""
    if construct_func == construct_dataset:
        X, y = construct_dataset(n_samples, kwargs.get('n_classes', 8))
    elif construct_func == construct_moons_dataset:
        X, y = construct_moons_dataset(n_samples, noise or 0.1)
    else:
        raise ValueError("Unknown dataset constructor")

    dataset = TorchDataset(X, y)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

def visualize_dataset(dataloader, n_samples=100, title="Dataset Visualization"):
    """Visualize samples from dataloader"""
    plt.figure(figsize=(8, 8))

    # Get some samples
    samples = []
    labels = []
    for X_batch, y_batch in dataloader:
        samples.append(X_batch)
        labels.append(y_batch)
        if len(samples) * dataloader.batch_size >= n_samples:
            break

    if samples:
        X_vis = torch.cat(samples, dim=0)[:n_samples].numpy()
        y_vis = torch.cat(labels, dim=0)[:n_samples].numpy()

        scatter = plt.scatter(X_vis[:, 0], X_vis[:, 1], c=y_vis, cmap='tab10', alpha=0.7)
        plt.colorbar(scatter)
        plt.title(title)
        plt.xlabel('X1')
        plt.ylabel('X2')
        plt.grid(True, alpha=0.3)
        plt.show()
    else:
        print("No samples to visualize")

def visualize_multiple_datasets(dataloaders, titles=None):
    """Visualize multiple datasets in subplots"""
    if titles is None:
        titles = [f"Dataset {i+1}" for i in range(len(dataloaders))]

    n_datasets = len(dataloaders)
    n_samples = 200
    fig, axes = plt.subplots(1, n_datasets, figsize=(4*n_datasets, 4))
    if n_datasets == 1:
        axes = [axes]

    for i, (dataloader, title) in enumerate(zip(dataloaders, titles)):
        samples = []
        labels = []
        for X_batch, y_batch in dataloader:
            samples.append(X_batch)
            labels.append(y_batch)
            if len(samples) * dataloader.batch_size >= n_samples:
                break

        if samples:
            X_vis = torch.cat(samples, dim=0)[:n_samples].numpy()
            y_vis = torch.cat(labels, dim=0)[:n_samples].numpy()

            scatter = axes[i].scatter(X_vis[:, 0], X_vis[:, 1], c=y_vis, cmap='tab10', alpha=0.7)
            axes[i].set_title(title)
            axes[i].set_xlabel('X1')
            axes[i].set_ylabel('X2')
            axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Example usage:
# dataloader = create_dataset_loader(construct_dataset, n_samples=100, n_classes=8, batch_size=32)
# visualize_dataset(dataloader, title="Rotating Beams Dataset")

# moons_dataloader = create_dataset_loader(construct_moons_dataset, n_samples=200, noise=0.1, batch_size=32)
# visualize_dataset(moons_dataloader, title="Moons Dataset")





# Пайплайн обучения

# Визуализация (рефакторинг - универсальные функции для до и после обучения)

## Утилитки

In [ ]:
# Утилита для извлечения датасета из DataLoader
def dataloader_to_numpy(dataloader, max_samples=None):
    """
    Конвертирует PyTorch DataLoader в numpy массивы.

    Parameters:
    -----------
    dataloader : DataLoader
        Торчевый датасет
    max_samples : int or None
        Максимальное количество семплов (None = все)

    Returns:
    --------
    X : array-like, shape (n_samples, 2)
        Координаты точек
    y : array-like, shape (n_samples,)
        Метки классов
    """
    X_list = []
    y_list = []

    for X_batch, y_batch in dataloader:
        X_list.append(X_batch)
        y_list.append(y_batch)
        if max_samples and sum(len(x) for x in X_list) >= max_samples:
            break

    X = torch.cat(X_list, dim=0).numpy()
    y = torch.cat(y_list, dim=0).numpy()

    if max_samples:
        X = X[:max_samples]
        y = y[:max_samples]

    return X, y


In [ ]:
# Универсальные функции визуализации векторного поля модели

def create_model_vector_field_func(pipeline, condition, timestamp, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    Создает функцию для вычисления НАПРАВЛЕНИЯ ДЕНОЙЗИНГА на сетке.

    Векторы показывают направление от зашумленных точек к предсказанным чистым данным.
    Это направление, куда модель "тянет" точки, чтобы удалить шум.

    Parameters:
    -----------
    pipeline : Pipeline
        Модель (обученная или необученная)
    condition : int or None
        Условие (класс), или None для unconditional (использует unk_cond_id)
    timestamp : int
        Таймстемп диффузии (точки сетки считаются зашумленными на этом timestep)
    device : str
        Устройство для вычислений

    Returns:
    --------
    vector_func : callable
        Функция, принимающая точки и возвращающая векторы НАПРАВЛЕНИЯ К ДАННЫМ
    """
    def vector_func(points):
        # Ensure points are on the correct device
        points_tensor = torch.tensor(points, dtype=torch.float32).to(device)

        # Таймстемп
        t_tensor = torch.tensor([timestamp] * points_tensor.shape[0], dtype=torch.long).to(device)

        # ИСПРАВЛЕНИЕ: НЕ добавляем шум! Считаем points уже зашумленными на timestep t
        # Подготовка condition: если None, используем unk_cond_id
        if condition is None:
            condition_val = pipeline.unk_cond_id
        else:
            condition_val = condition

        # Ensure condition tensor is on the correct device
        condition_tensor = torch.tensor([condition_val] * points_tensor.shape[0], device=device)

        # Получаем предсказание модели (предсказывает шум)
        with torch.no_grad():
            pred_noise = pipeline(points_tensor,
                                condition=condition_tensor,
                                timestamp=t_tensor)

        # Вычисляем направление денойзинга (куда модель "тянет" точки к данным)
        sqrt_alpha_bar = pipeline.noise_schedule.sqrt_alphas_cumprod[timestamp]
        sqrt_one_minus_alpha_bar = pipeline.noise_schedule.sqrt_one_minus_alphas_cumprod[timestamp]

        # Pred x_0 calculation should also be on the correct device
        pred_x0 = (points_tensor - sqrt_one_minus_alpha_bar * pred_noise) / sqrt_alpha_bar

        # Направление от текущей точки к предсказанным чистым данным
        denoising_direction = pred_x0 - points_tensor

        return denoising_direction.cpu().numpy()

    return vector_func

In [ ]:
def plot_model_vectors_static(X, y, pipeline, condition=0, timestamp=0,
                              normalize_lengths=True, grid_resolution=15,
                              title=None, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    Статическая визуализация векторного поля модели для заданного условия и таймстемпа.

    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Координаты точек датасета
    y : array-like, shape (n_samples,)
        Метки классов
    pipeline : Pipeline
        Модель (обученная или необученная)
    condition : int
        Условие (класс)
    timestamp : int
        Таймстемп
    normalize_lengths : bool
        Нормировать длины векторов
    grid_resolution : int
        Разрешение сетки для векторного поля
    title : str or None
        Заголовок графика
    device : str
        Устройство для вычислений
    """
    if title is None:
        title = f"Model Vector Field - Condition {condition}, Timestamp {timestamp}"

    # Create vector function with the correct device
    vector_func = create_model_vector_field_func(pipeline, condition, timestamp, device)

    # Use plot_vectors function with vector_func
    plot_vectors(X, y, vector_func=vector_func, grid_resolution=grid_resolution,
                normalize_lengths=normalize_lengths, title=title)

In [ ]:
def plot_model_vectors_interactive(X, y, pipeline, device='cuda' if torch.cuda.is_available() else 'cpu',
                                   grid_resolution=30, normalize_lengths=True):
    """
    Интерактивная визуализация векторного поля модели с ползунками.
    Работает как с обученной, так и с необученной моделью.

    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Координаты точек датасета
    y : array-like, shape (n_samples,)
        Метки классов
    pipeline : Pipeline
        Модель (обученная или необученная)
    device : str
        Устройство для вычислений
    grid_resolution : int
        Разрешение сетки для векторного поля
    normalize_lengths : bool
        Нормировать длины векторов
    """
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output

        # Ползунки для параметров
        condition_slider = widgets.IntSlider(
            value=0,
            min=-1,  # -1 означает unconditional
            max=len(np.unique(y)) - 1,
            step=1,
            description='Condition:'
        )

        timestamp_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=pipeline.noise_schedule.num_steps - 1,
            step=1,
            description='Timestamp:'
        )

        resolution_slider = widgets.IntSlider(
            value=grid_resolution,
            min=5,
            max=100,
            step=5,
            description='Resolution:'
        )

        normalize_checkbox = widgets.Checkbox(
            value=normalize_lengths,
            description='Normalize lengths'
        )

        # Функция для обновления графика
        def update_plot(condition, timestamp, resolution, normalize):
            clear_output(wait=True)

            # Если condition == -1, используем None (unconditional)
            cond_val = None if condition == -1 else condition

            # Создаем векторную функцию
            vector_func = create_model_vector_field_func(pipeline, cond_val, timestamp, device)

            # Используем функцию plot_vectors с vector_func
            plt.figure(figsize=(10, 8))
            cond_label = "Unconditional" if cond_val is None else f"Condition {cond_val}"


            sample_points = torch.tensor(X, dtype=torch.float32)
            t_tensor = torch.tensor([timestamp] * sample_points.shape[0], dtype=torch.long)
            noised_samples, _ = pipeline.noise_schedule.add_noise(sample_points, t_tensor)

            plot_vectors(noised_samples.cpu().numpy(), y, vector_func=vector_func, grid_resolution=resolution,
                        normalize_lengths=normalize,
                        title=f"Model Vector Field - {cond_label}, Timestamp {timestamp}")
            plt.show()

        # Создаем интерактивный виджет
        interactive_plot = widgets.interactive(update_plot,
                                              condition=condition_slider,
                                              timestamp=timestamp_slider,
                                              resolution=resolution_slider,
                                              normalize=normalize_checkbox)
        display(interactive_plot)

    except ImportError:
        print("ipywidgets не установлен. Установите его для интерактивной визуализации.")
        print("pip install ipywidgets")
        print("\nИспользуйте plot_model_vectors_static для статической визуализации.")


In [ ]:
def plot_noise_stages_interactive(X, y, noise_schedule, num_stages=5):
    """
    Интерактивная визуализация датасета на разных этапах зашумления.

    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Координаты точек
    y : array-like, shape (n_samples,)
        Метки классов
    noise_schedule : NoiseSchedule
        Объект расписания шума
    num_stages : int
        Количество этапов зашумления для отображения
    """
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output

        # Ползунок для таймстемпа
        timestamp_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=noise_schedule.num_steps - 1,
            step=1,
            description='Timestamp:'
        )

        def update_plot(timestamp):
            clear_output(wait=True)

            sample_points = torch.tensor(X, dtype=torch.float32)
            t_tensor = torch.tensor([timestamp] * sample_points.shape[0], dtype=torch.long)
            noised_samples, _ = noise_schedule.add_noise(sample_points, t_tensor)

            plt.figure(figsize=(8, 6))
            plot_dataset(noised_samples.cpu().numpy(), y,
                        title=f'Dataset at Timestep {timestamp}')
            plt.show()

        interactive_plot = widgets.interactive(update_plot, timestamp=timestamp_slider)
        display(interactive_plot)

    except ImportError:
        print("ipywidgets не установлен. Используйте plot_noise_stages для статической визуализации.")


def plot_noise_stages(X, y, noise_schedule, timesteps=None, num_stages=5,
                      title="Dataset at Different Noise Stages", figsize=(16, 4)):
    """
    Визуализирует датасет на разных этапах зашумления (статическая версия).

    Parameters:
    -----------
    X : array-like, shape (n_samples, 2)
        Координаты точек
    y : array-like, shape (n_samples,)
        Метки классов
    noise_schedule : NoiseSchedule
        Объект расписания шума
    timesteps : list or None
        Список таймстемпов для визуализации (если None, создается автоматически)
    num_stages : int
        Количество этапов (если timesteps=None)
    title : str
        Заголовок графика
    figsize : tuple
        Размер фигуры
    """
    if timesteps is None:
        # Создаем равномерно распределенные таймстемпы
        max_t = noise_schedule.num_steps - 1
        if num_stages > 1:
            timesteps = [int(i * max_t / (num_stages - 1)) for i in range(num_stages)]
        else:
            timesteps = [0]

    sample_points = torch.tensor(X, dtype=torch.float32)

    fig, axes = plt.subplots(1, len(timesteps), figsize=(int(figsize[0]*(num_stages/4)), figsize[1]))
    if len(timesteps) == 1:
        axes = [axes]

    for i, t in enumerate(timesteps):
        t_tensor = torch.tensor([t] * sample_points.shape[0], dtype=torch.long)
        noised_samples, _ = noise_schedule.add_noise(sample_points, t_tensor)

        plot_dataset(noised_samples.cpu().numpy(), y, title=f'Timestep {t}',
                    figsize=(int(figsize[0]*(num_stages/4))//len(timesteps), figsize[1]), ax=axes[i])

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


## Обучение и визуализация

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Глобальные параметры
dim = 64  # 2D points
num_heads = 8
condition_dim = 64 # Example: if using class labels as conditions
timestamp_dim = 64 # Example: for time embedding
num_layers = 8
GLOB_N_CLASSES = 4
GLOB_N_TIMESTAMPS = 1000
GLOB_SAMPLES_PER_CLASS = 500
max_condition = GLOB_N_CLASSES # Should be at least max(y) + 1 #3 clasess
max_timestamp = GLOB_N_TIMESTAMPS # Should be num_steps for noise schedule
GL_BETA_START = 1e-5
GL_BETA_END = 1e-3

# Создаем модель (до обучения)
pipeline = Pipeline(dim, num_heads, condition_dim, timestamp_dim, num_layers,
                    scheduler_type='linear',
                    max_condition=max_condition, max_timestamp=max_timestamp,
                    act_layer=nn.GELU, norm_layer=nn.LayerNorm, # Added missing arguments
                    device=device)

# Создаем датасет и даталоадер для обучения
batch_size = 32
dataloader = create_dataset_loader(construct_dataset, n_samples=GLOB_SAMPLES_PER_CLASS,
                                   n_classes=GLOB_N_CLASSES, batch_size=batch_size)

# Извлекаем данные из даталоадера для визуализации
X_train, y_train = dataloader_to_numpy(dataloader, max_samples=GLOB_SAMPLES_PER_CLASS * GLOB_N_CLASSES)

print(f"Dataset shape: X={X_train.shape}, y={y_train.shape}")
print(f"Device: {device}")

In [ ]:
# ============================================
# ВИЗУАЛИЗАЦИЯ ДО ОБУЧЕНИЯ
# ============================================

pipeline.to('cpu')
print("=" * 60)
print("ВИЗУАЛИЗАЦИЯ ДО ОБУЧЕНИЯ")
print("=" * 60)

# 1. Базовая визуализация датасета
plot_dataset(X_train, y_train, title="Generated Dataset (Training Data)")

# 2. Плотность распределения
plot_density_analytical(X_train, y_train, title="Density Heatmap of Training Dataset")

# 3. Визуализация этапов зашумления (статическая)
plot_noise_stages(X_train, y_train, pipeline.noise_schedule, num_stages=8,
                  title="Dataset at Different Noise Stages")

# 4. Интерактивная визуализация зашумления (если доступны ipywidgets)
print("\nИнтерактивная визуализация зашумления:")
plot_noise_stages_interactive(X_train, y_train, pipeline.noise_schedule)

# 5. Статическая визуализация векторного поля необученной модели
print("\nВекторное поле необученной модели (статическая визуализация):")
plot_model_vectors_static(X_train, y_train, pipeline, condition=0, timestamp=50,
                          title="Untrained Model - Condition 0, Timestamp 50")

# 6. Интерактивная визуализация векторного поля необученной модели
print("\nИнтерактивная визуализация векторного поля необученной модели:")
plot_model_vectors_interactive(X_train, y_train, pipeline, device=device)


In [ ]:
# ============================================
# ОБУЧЕНИЕ МОДЕЛИ
# ============================================

print("\n" + "=" * 60)
print("ОБУЧЕНИЕ МОДЕЛИ С UNCONDITIONAL TRAINING")
print("=" * 60)

num_epochs = 80
lr = 2e-3
uncond_prob = 0.15  # 15% вероятность unconditional training
visualize_every = 5  # Визуализация каждые 10 эпох


pipeline, optimizer, scheduler = prepare_train_pipeline(pipeline, dataloader, num_epochs, lr, device)

print(f"\nПараметры обучения:")
print(f"  - Эпохи: {num_epochs}")
print(f"  - Learning rate: {lr}")
print(f"  - Unconditional probability: {uncond_prob}")
print(f"  - Визуализация каждые {visualize_every} эпох")
print(f"\nНачинаем обучение...\n")

train_pipeline(pipeline, optimizer, scheduler, dataloader, num_epochs, device,
              uncond_prob=uncond_prob, visualize_every=visualize_every, vis_sample_size=200)

print("\n✅ Обучение завершено!")

In [ ]:
# ============================================ ВИЗУАЛИЗАЦИЯ ПОСЛЕ ОБУЧЕНИЯ
# ============================================

print("\n" + "=" * 60)
print("ВИЗУАЛИЗАЦИЯ ПОСЛЕ ОБУЧЕНИЯ")
print("=" * 60)
"""
# 1. Сравнение Unconditional vs Conditional
print("\n1. Сравнение Unconditional vs Conditional (timestamp=50):")
fig, axes = plt.subplots(1, GLOB_N_CLASSES + 1, figsize=(5 * (GLOB_N_CLASSES + 1), 4))

# Unconditional
plt.sca(axes[0])
plot_model_vectors_static(X_train, y_train, pipeline, condition=None, timestamp=50,
                          grid_resolution=12,
                          title="Unconditional")

# Conditional для каждого класса
for i in range(GLOB_N_CLASSES):
    plt.sca(axes[i + 1])
    plot_model_vectors_static(X_train, y_train, pipeline, condition=i, timestamp=50,
                              grid_resolution=12,
                              title=f"Condition {i}")
plt.tight_layout()
plt.show()

# 2. Unconditional на разных таймстемпах
print("\n2. Unconditional модель на разных таймстемпах:")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, t in enumerate([25, 50, 75]):
    plt.sca(axes[idx])
    plot_model_vectors_static(X_train, y_train, pipeline, condition=None, timestamp=t,
                              grid_resolution=12,
                              title=f"Unconditional, t={t}")
plt.tight_layout()
plt.show()
"""
# 3. Интерактивная визуализация векторного поля обученной модели
print("\n3. Интерактивная визуализация (используйте Condition=-1 для unconditional):")
plot_model_vectors_interactive(X_train, y_train, pipeline, device=device)

# 4. Визуализация этапов зашумления
print("\n4. Этапы зашумления датасета:")
plot_noise_stages(X_train, y_train, pipeline.noise_schedule, num_stages=5,
                  title="Dataset at Different Noise Stages")

# 5. Интерактивная визуализация зашумления
print("\n5. Интерактивная визуализация зашумления:")
plot_noise_stages_interactive(X_train, y_train, pipeline.noise_schedule)

In [ ]:
# 6. Интерактивная визуализация векторного поля необученной модели
print("\nИнтерактивная визуализация векторного поля необученной модели:")
plot_model_vectors_interactive(X_train, y_train, pipeline, device=device)


## Резюме обновлений

### 1. Универсальные функции визуализации
Созданы функции, которые работают как с необученной, так и с обученной моделью:
- `dataloader_to_numpy()` - конвертация из DataLoader в numpy
- `plot_model_vectors_static()` - статическая визуализация
- `plot_model_vectors_interactive()` - интерактивная с ползунками
- `plot_noise_stages()` / `plot_noise_stages_interactive()` - визуализация зашумления

### 2. Unconditional Training
Обновлена функция `train_pipeline()` с поддержкой:
- **uncond_prob** - вероятность unconditional обучения (рандомно заменяет condition на unk_cond_id)
- **visualize_every** - визуализация прогресса каждые N эпох
- **vis_sample_size** - размер выборки для визуализации

### 3. Визуализация прогресса обучения
Добавлена функция `visualize_training_progress_fast()`:
- Быстрая визуализация (маленькая сетка, полностью на PyTorch)
- Показывает 3 графика: unconditional модель с разными таймстемпами
- Автоматически вызывается во время обучения

### 4. Поддержка Unconditional в интерактивной визуализации
- В ползунке Condition: -1 = Unconditional, 0+ = Conditional
- Автоматическое определение и использование unk_cond_id

### Использование:

```python
# Обучение с unconditional
train_pipeline(pipeline, optimizer, scheduler, dataloader, num_epochs, device,
              uncond_prob=0.15,        # 15% unconditional примеров
              visualize_every=10,      # Визуализация каждые 10 эпох
              vis_sample_size=200)     # 200 точек для визуализации

# Визуализация unconditional модели
plot_model_vectors_static(X, y, pipeline, condition=None, timestamp=50)
# или интерактивно: двигайте ползунок Condition в -1
plot_model_vectors_interactive(X, y, pipeline, device=device)
```


## Краткое руководство по использованию

### Что было сделано в этом рефакторинге:

1. **Универсальная визуализация** - одни и те же функции работают до и после обучения
2. **Unconditional training** - модель учится работать без условий (condition)
3. **Визуализация прогресса** - автоматические графики во время обучения
4. **Интерактивность** - ползунки для всех параметров

### Ключевые параметры:

**train_pipeline():**
- `uncond_prob=0.15` - 15% примеров будут unconditional
- `visualize_every=10` - показывать графики каждые 10 эпох
- `vis_sample_size=200` - количество точек для визуализации

**Интерактивная визуализация:**
- Condition: -1 = Unconditional, 0+ = конкретный класс
- Timestamp: 0-99 (этап диффузии)
- Resolution: 5-30 (детализация сетки)
- Normalize: вкл/выкл нормализацию векторов

### Основные функции:

```python
# Конвертация данных
X, y = dataloader_to_numpy(dataloader)

# Статическая визуализация
plot_model_vectors_static(X, y, pipeline, condition=None, timestamp=50)

# Интерактивная визуализация
plot_model_vectors_interactive(X, y, pipeline, device=device)

# Обучение с визуализацией
train_pipeline(pipeline, optimizer, scheduler, dataloader,
               num_epochs, device, uncond_prob=0.15, visualize_every=10)
```


## ВАЖНОЕ ИСПРАВЛЕНИЕ: Визуализация векторного поля

### Проблема
Векторы показывали предсказанный шум и все указывали в одну сторону, не отражая структуру данных.

### Причина
Было двойное зашумление:
1. Берем точки сетки
2. Добавляем к ним рандомный шум
3. Модель предсказывает этот рандомный шум
4. Результат: бессмысленные векторы

### Решение
Теперь векторы показывают НАПРАВЛЕНИЕ ДЕНОЙЗИНГА:
- Точки сетки считаются УЖЕ зашумленными на timestep t
- Модель предсказывает где должны быть чистые данные
- Векторы = направление от зашумленной точки к предсказанным чистым данным

### Формула
```
pred_x0 = (x_t - sqrt(1-alpha_bar_t) * epsilon_pred) / sqrt(alpha_bar_t)
denoising_direction = pred_x0 - x_t
```

### Результат
Векторы теперь указывают на области с реальным распределением данных!


## Анимация процесса денойзинга

Визуализация того, как случайный шум превращается в структурированные данные


In [ ]:
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from scipy.interpolate import interp1d
import os

def ddpm_sampling_with_history(pipeline, n_samples, condition, device, num_steps=None, eta=0.0):
    """
    DDPM/DDIM sampling с сохранением истории. eta=0 для DDIM, eta=1 для DDPM.

    Parameters:
    -----------
    pipeline : Pipeline
        Обученная модель
    n_samples : int
        Количество точек
    condition : torch.Tensor or None
        Условия [n_samples] или None
    device : str
        Устройство
    num_steps : int or None
        Количество шагов (если None, все timesteps)
    eta : float
        Стохастичность (0=DDIM, 1=DDPM)

    Returns:
    --------
    history : torch.Tensor
        [num_steps+1, n_samples, 2]
    """
    pipeline.eval()
    num_steps_model = pipeline.noise_schedule.num_steps
    if num_steps is None:
        num_steps = num_steps_model

    x = torch.randn(n_samples, 2, device=device)*0.8
    if condition is None:
        condition_tensor = torch.full((n_samples,), pipeline.unk_cond_id, dtype=torch.long, device=device)
    else:
        condition_tensor = condition.to(device)

    history = [x.clone().cpu()]
    timesteps = torch.linspace(num_steps_model - 1, 0, num_steps, dtype=torch.float, device=device).round().long()

    with torch.no_grad():
        for i, t in enumerate(timesteps):
            t_batch = torch.full((n_samples,), t, dtype=torch.long, device=device)
            pred_noise = pipeline(x, condition=condition_tensor, timestamp=t_batch)

            alpha = pipeline.noise_schedule.alphas[t]
            alpha_bar = pipeline.noise_schedule.alphas_cumprod[t]

            # Определяем t_prev и alpha_bar_prev
            if i + 1 < len(timesteps):
                t_prev = timesteps[i + 1]
                alpha_bar_prev = pipeline.noise_schedule.alphas_cumprod[t_prev]
            else:
                t_prev = 0
                alpha_bar_prev = torch.tensor(1.0, device=device)

            # Предсказываем x_0
            pred_x0 = (x - torch.sqrt(1 - alpha_bar) * pred_noise) / torch.sqrt(alpha_bar)

            # DDIM шаг
            dir_x0 = torch.sqrt(alpha_bar_prev) * pred_x0
            dir_xt = torch.sqrt(1 - alpha_bar_prev) * pred_noise
            x = dir_x0 + dir_xt

            # Добавляем шум, если не последний шаг и eta > 0
            if i + 1 < len(timesteps) and eta > 0:
                noise = torch.randn_like(x)
                sigma = eta * torch.sqrt((1 - alpha_bar_prev) / (1 - alpha_bar) * (1 - alpha_bar / alpha_bar_prev))
                x = x + sigma * noise

            history.append(x.clone().cpu())

    pipeline.train()
    return torch.stack(history)

def interpolate_trajectory(history, interpolation_factor=5):
    """
    Кубическая интерполяция траекторий точек.

    Parameters:
    -----------
    history : torch.Tensor
        История состояний, shape: [num_steps, n_samples, 2]
    interpolation_factor : int
        Во сколько раз увеличить количество кадров

    Returns:
    --------
    interpolated : torch.Tensor
        Интерполированная история, shape: [num_steps * interpolation_factor, n_samples, 2]
    """
    num_steps, n_samples, dim = history.shape

    # Исходные временные точки
    t_original = np.linspace(0, 1, num_steps)

    # Новые временные точки (с интерполяцией)
    t_new = np.linspace(0, 1, (num_steps - 1) * interpolation_factor + 1)

    # Интерполируем каждую координату каждой точки
    interpolated = np.zeros((len(t_new), n_samples, dim))

    for i in range(n_samples):
        for d in range(dim):
            # Кубическая интерполяция
            f = interp1d(t_original, history[:, i, d].numpy(), kind='cubic')
            interpolated[:, i, d] = f(t_new)

    return torch.from_numpy(interpolated).float()


def create_denoising_animation(pipeline, n_samples_per_class=50, device='cuda',
                               num_denoising_steps=50, interpolation_factor=5,
                               fps=30, filename='denoising_animation.gif',
                               figsize=(10, 10), dpi=100):
    """
    Создает анимацию процесса денойзинга с интерполяцией.

    Parameters:
    -----------
    pipeline : Pipeline
        Обученная модель
    n_samples_per_class : int
        Количество точек на каждый класс
    device : str
        Устройство
    num_denoising_steps : int
        Количество шагов денойзинга
    interpolation_factor : int
        Фактор интерполяции между кадрами
    fps : int
        Кадров в секунду
    filename : str
        Имя файла для сохранения
    figsize : tuple
        Размер фигуры
    dpi : int
        DPI для сохранения
    """
    print(f"Генерация траекторий денойзинга...")

    # Определяем количество классов
    n_classes = pipeline.unk_cond_id  # Все классы кроме unk
    n_total = n_samples_per_class * n_classes

    # Создаем условия для каждого класса
    conditions = torch.arange(n_classes, device=device).repeat_interleave(n_samples_per_class)

    # Генерируем траектории
    history = ddpm_sampling_with_history(pipeline, n_total, conditions, device,
                                         num_steps=num_denoising_steps)

    print(f"Интерполяция траекторий (фактор {interpolation_factor})...")
    # Интерполируем траектории
    history_interp = interpolate_trajectory(history, interpolation_factor)

    num_frames = history_interp.shape[0]
    print(f"Создание анимации ({num_frames} кадров)...")

    # Создаем фигуру
    fig, ax = plt.subplots(figsize=figsize, facecolor='black')
    ax.set_facecolor('black')

    # Определяем границы
    all_points = history_interp.reshape(-1, 2).numpy()
    x_min, x_max = all_points[:, 0].min() - 1, all_points[:, 0].max() + 1
    y_min, y_max = all_points[:, 1].min() - 1, all_points[:, 1].max() + 1

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2, color='white')
    ax.tick_params(colors='white')

    # Цвета для каждого класса
    colors = plt.cm.Set3(np.linspace(0, 1, n_classes))

    # Создаем scatter plot для каждого класса
    scatters = []
    for class_idx in range(n_classes):
        mask = (conditions.cpu() == class_idx).numpy()
        scatter = ax.scatter([], [], c=[colors[class_idx]], s=50, alpha=0.8,
                           edgecolors='white', linewidths=0.5,
                           label=f'Class {class_idx}')
        scatters.append(scatter)

    # Добавляем легенду
    legend = ax.legend(facecolor='black', edgecolor='white',
                      loc='upper right', framealpha=0.8)
    for text in legend.get_texts():
        text.set_color('white')

    # Текст с прогрессом
    progress_text = ax.text(0.02, 0.98, '', transform=ax.transAxes,
                           color='white', fontsize=12, verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

    def init():
        for scatter in scatters:
            scatter.set_offsets(np.empty((0, 2)))
        progress_text.set_text('')
        return scatters + [progress_text]

    def animate(frame):
        # Текущее состояние точек
        points = history_interp[frame].numpy()

        # Обновляем каждый scatter для каждого класса
        for class_idx, scatter in enumerate(scatters):
            mask = (conditions.cpu() == class_idx).numpy()
            class_points = points[mask]
            scatter.set_offsets(class_points)

        # Обновляем текст прогресса
        progress = frame / (num_frames - 1) * 100
        timestep = int((1 - frame / (num_frames - 1)) * (num_denoising_steps - 1))
        progress_text.set_text(f'Progress: {progress:.1f}%\nTimestep: {timestep}')

        return scatters + [progress_text]

    # Создаем анимацию
    anim = FuncAnimation(fig, animate, init_func=init, frames=num_frames,
                        interval=1000/fps, blit=True, repeat=True)

    # Определяем формат по расширению файла
    file_ext = os.path.splitext(filename)[1].lower()

    # Сохраняем
    print(f"Сохранение анимации в {filename}...")

    if file_ext in ['.mp4', '.avi', '.mov']:
        # Видео форматы
        try:
            writer = FFMpegWriter(fps=fps, bitrate=2000, codec='libx264')
            anim.save(filename, writer=writer, dpi=dpi)
            print(f"✅ Видео сохранено: {filename}")
        except Exception as e:
            print(f"⚠️ Ошибка сохранения видео: {e}")
            print("Попробуйте установить ffmpeg: conda install ffmpeg или pip install ffmpeg-python")
            # Fallback на GIF
            fallback_name = filename.rsplit('.', 1)[0] + '.gif'
            print(f"Сохранение в GIF: {fallback_name}")
            writer = PillowWriter(fps=fps)
            anim.save(fallback_name, writer=writer, dpi=dpi)
            print(f"✅ GIF сохранен: {fallback_name}")
    else:
        # GIF формат (по умолчанию)
        writer = PillowWriter(fps=fps)
        anim.save(filename, writer=writer, dpi=dpi)
        print(f"✅ GIF сохранен: {filename}")

    plt.close(fig)

    return anim


In [ ]:
# Создаем анимацию процесса денойзинга
# Параметры можно настроить под ваши нужды

pipeline.to('cpu')
# Вариант 1: MP4 (рекомендуется - меньше размер, выше качество)
create_denoising_animation(
    pipeline=pipeline,
    n_samples_per_class=20,      # Количество точек на каждый класс
    device='cpu',
    num_denoising_steps=100,      # Количество шагов денойзинга (меньше = быстрее)
    interpolation_factor=2,      # Фактор интерполяции (больше = плавнее)
    fps=30,                      # Кадров в секунду
    filename='denoising_animation_c4.gif',  # MP4, GIF, AVI, или MOV
    figsize=(10, 10),
    dpi=60
)

# Вариант 2: GIF (если нет ffmpeg)
# create_denoising_animation(
#     pipeline=pipeline,
#     n_samples_per_class=50,
#     device=device,
#     num_denoising_steps=50,
#     interpolation_factor=5,
#     fps=30,
#     filename='denoising_animation.gif',
#     figsize=(10, 10),
#     dpi=100
# )


### КФГ семплинг

In [ ]:
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from scipy.interpolate import interp1d
import os


def ddpm_cfg_sampling_with_history(pipeline, n_samples, condition, device, num_steps=None, eta=0.01, cfg=1.8, cfg_start=0.8):
    """
    DDPM/DDIM sampling с сохранением истории. eta=0 для DDIM, eta=1 для DDPM.

    Parameters:
    -----------
    pipeline : Pipeline
        Обученная модель
    n_samples : int
        Количество точек
    condition : torch.Tensor or None
        Условия [n_samples] или None
    device : str
        Устройство
    num_steps : int or None
        Количество шагов (если None, все timesteps)
    eta : float
        Стохастичность (0=DDIM, 1=DDPM)

    Returns:
    --------
    history : torch.Tensor
        [num_steps+1, n_samples, 2]
    """
    pipeline.eval()
    num_steps_model = pipeline.noise_schedule.num_steps
    if num_steps is None:
        num_steps = num_steps_model

    x = torch.randn(n_samples, 2, device=device)*0.8
    if condition is None:
        condition_tensor = torch.full((n_samples,), pipeline.unk_cond_id, dtype=torch.long, device=device)
    else:
        uncondition_tensor = torch.full((n_samples,), pipeline.unk_cond_id, dtype=torch.long, device=device)
        condition_tensor = condition.to(device)

    history = [x.clone().cpu()]
    timesteps = torch.linspace(num_steps_model - 1, 0, num_steps, dtype=torch.float, device=device).round().long()

    with torch.no_grad():
        for i, t in enumerate(timesteps):
            t_batch = torch.full((n_samples,), t, dtype=torch.long, device=device)
            pred_noise = pipeline(x, condition=condition_tensor, timestamp=t_batch)

            if not condition is None:
                if cfg_start*num_steps >= i and i>2:
                    pred_noise_neg = pipeline(x, condition=uncondition_tensor, timestamp=t_batch)
                    pred_noise = pred_noise_neg*(1-cfg)+pred_noise*cfg

            alpha = pipeline.noise_schedule.alphas[t]
            alpha_bar = pipeline.noise_schedule.alphas_cumprod[t]

            # Определяем t_prev и alpha_bar_prev
            if i + 1 < len(timesteps):
                t_prev = timesteps[i + 1]
                alpha_bar_prev = pipeline.noise_schedule.alphas_cumprod[t_prev]
            else:
                t_prev = 0
                alpha_bar_prev = torch.tensor(1.0, device=device)

            # Предсказываем x_0
            pred_x0 = (x - torch.sqrt(1 - alpha_bar) * pred_noise) / torch.sqrt(alpha_bar)

            # DDIM шаг
            dir_x0 = torch.sqrt(alpha_bar_prev) * pred_x0
            dir_xt = torch.sqrt(1 - alpha_bar_prev) * pred_noise
            x = dir_x0 + dir_xt

            # Добавляем шум, если не последний шаг и eta > 0
            if i + 1 < len(timesteps) and eta > 0:
                noise = torch.randn_like(x)
                sigma = eta * torch.sqrt((1 - alpha_bar_prev) / (1 - alpha_bar) * (1 - alpha_bar / alpha_bar_prev))
                x = x + sigma * noise

            history.append(x.clone().cpu())

    pipeline.train()
    return torch.stack(history)



def create_denoising_cfg_animation(pipeline, n_samples_per_class=50, device='cuda',
                               num_denoising_steps=50, interpolation_factor=5,
                               fps=30, filename='denoising_animation.gif',
                               figsize=(10, 10), dpi=100):
    """
    Создает анимацию процесса денойзинга с интерполяцией.

    Parameters:
    -----------
    pipeline : Pipeline
        Обученная модель
    n_samples_per_class : int
        Количество точек на каждый класс
    device : str
        Устройство
    num_denoising_steps : int
        Количество шагов денойзинга
    interpolation_factor : int
        Фактор интерполяции между кадрами
    fps : int
        Кадров в секунду
    filename : str
        Имя файла для сохранения
    figsize : tuple
        Размер фигуры
    dpi : int
        DPI для сохранения
    """
    print(f"Генерация траекторий денойзинга...")

    # Определяем количество классов
    n_classes = pipeline.unk_cond_id  # Все классы кроме unk
    n_total = n_samples_per_class * n_classes

    # Создаем условия для каждого класса
    conditions = torch.arange(n_classes, device=device).repeat_interleave(n_samples_per_class)

    # Генерируем траектории
    history = ddpm_cfg_sampling_with_history(pipeline, n_total, conditions, device,
                                         num_steps=num_denoising_steps)

    print(f"Интерполяция траекторий (фактор {interpolation_factor})...")
    # Интерполируем траектории
    hist_len = history.shape[0]
    hist_len_b = int(0.9*hist_len)

    history_interp_1 = interpolate_trajectory(history[:hist_len_b], interpolation_factor)
    history_interp_2 = interpolate_trajectory(history[hist_len_b:], interpolation_factor*2)
    history_interp = torch.cat([history_interp_1, history_interp_2])


    num_frames = history_interp.shape[0]
    print(f"Создание анимации ({num_frames} кадров)...")

    # Создаем фигуру
    fig, ax = plt.subplots(figsize=figsize, facecolor='black')
    ax.set_facecolor('black')

    # Определяем границы
    all_points = history_interp.reshape(-1, 2).numpy()
    x_min, x_max = all_points[:, 0].min() - 1, all_points[:, 0].max() + 1
    y_min, y_max = all_points[:, 1].min() - 1, all_points[:, 1].max() + 1

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2, color='white')
    ax.tick_params(colors='white')

    # Цвета для каждого класса
    colors = plt.cm.Set3(np.linspace(0, 1, n_classes))

    # Создаем scatter plot для каждого класса
    scatters = []
    for class_idx in range(n_classes):
        mask = (conditions.cpu() == class_idx).numpy()
        scatter = ax.scatter([], [], c=[colors[class_idx]], s=50, alpha=0.8,
                           edgecolors='white', linewidths=0.5,
                           label=f'Class {class_idx}')
        scatters.append(scatter)

    # Добавляем легенду
    legend = ax.legend(facecolor='black', edgecolor='white',
                      loc='upper right', framealpha=0.8)
    for text in legend.get_texts():
        text.set_color('white')

    # Текст с прогрессом
    progress_text = ax.text(0.02, 0.98, '', transform=ax.transAxes,
                           color='white', fontsize=12, verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

    def init():
        for scatter in scatters:
            scatter.set_offsets(np.empty((0, 2)))
        progress_text.set_text('')
        return scatters + [progress_text]

    def animate(frame):
        # Текущее состояние точек
        points = history_interp[frame].numpy()

        # Обновляем каждый scatter для каждого класса
        for class_idx, scatter in enumerate(scatters):
            mask = (conditions.cpu() == class_idx).numpy()
            class_points = points[mask]
            scatter.set_offsets(class_points)

        # Обновляем текст прогресса
        progress = frame / (num_frames - 1) * 100
        timestep = int((1 - frame / (num_frames - 1)) * (num_denoising_steps - 1))
        progress_text.set_text(f'Progress: {progress:.1f}%\nTimestep: {timestep}')

        return scatters + [progress_text]

    # Создаем анимацию
    anim = FuncAnimation(fig, animate, init_func=init, frames=num_frames,
                        interval=1000/fps, blit=True, repeat=True)

    # Определяем формат по расширению файла
    file_ext = os.path.splitext(filename)[1].lower()

    # Сохраняем
    print(f"Сохранение анимации в {filename}...")

    if file_ext in ['.mp4', '.avi', '.mov']:
        # Видео форматы
        try:
            writer = FFMpegWriter(fps=fps, bitrate=2000, codec='libx264')
            anim.save(filename, writer=writer, dpi=dpi)
            print(f"✅ Видео сохранено: {filename}")
        except Exception as e:
            print(f"⚠️ Ошибка сохранения видео: {e}")
            print("Попробуйте установить ffmpeg: conda install ffmpeg или pip install ffmpeg-python")
            # Fallback на GIF
            fallback_name = filename.rsplit('.', 1)[0] + '.gif'
            print(f"Сохранение в GIF: {fallback_name}")
            writer = PillowWriter(fps=fps)
            anim.save(fallback_name, writer=writer, dpi=dpi)
            print(f"✅ GIF сохранен: {fallback_name}")
    else:
        # GIF формат (по умолчанию)
        writer = PillowWriter(fps=fps)
        anim.save(filename, writer=writer, dpi=dpi)
        print(f"✅ GIF сохранен: {filename}")

    plt.close(fig)

    return anim


In [ ]:
create_denoising_cfg_animation(
    pipeline=pipeline,
    n_samples_per_class=20,      # Количество точек на каждый класс
    device=device,
    num_denoising_steps=150,      # Количество шагов денойзинга (меньше = быстрее)
    interpolation_factor=2,      # Фактор интерполяции (больше = плавнее)
    fps=30,                      # Кадров в секунду
    filename='denoising_cfg_animation_c4_18.gif',  # MP4, GIF, AVI, или MOV
    figsize=(10, 10),
    dpi=60
)


### Дополнительные варианты анимации

Можно создать разные варианты с различными параметрами


In [ ]:
# Вариант 1: Быстрая анимация (меньше шагов и точек)
# create_denoising_animation(
#     pipeline=pipeline,
#     n_samples_per_class=30,
#     device=device,
#     num_denoising_steps=25,
#     interpolation_factor=3,
#     fps=20,
#     filename='denoising_fast.mp4',
#     figsize=(8, 8),
#     dpi=80
# )

# Вариант 2: Высококачественная анимация (больше точек, выше разрешение)
# create_denoising_animation(
#     pipeline=pipeline,
#     n_samples_per_class=100,
#     device=device,
#     num_denoising_steps=100,
#     interpolation_factor=10,
#     fps=60,
#     filename='denoising_hq.mp4',  # MP4 для HD качества!
#     figsize=(12, 12),
#     dpi=150
# )

# Вариант 3: Медленная детальная анимация
# create_denoising_animation(
#     pipeline=pipeline,
#     n_samples_per_class=50,
#     device=device,
#     num_denoising_steps=100,
#     interpolation_factor=8,
#     fps=15,  # Медленнее
#     filename='denoising_slow.mp4',
#     figsize=(10, 10),
#     dpi=100
# )


### Анимация с траекториями

Вариант с отображением траекторий движения точек


In [ ]:
def create_denoising_animation_with_trails(pipeline, n_samples_per_class=50, device='cpu',
                                          num_denoising_steps=50, interpolation_factor=5,
                                          fps=30, filename='denoising_trails.gif',
                                          figsize=(10, 10), dpi=100, trail_length=20):
    """
    Создает анимацию с траекториями движения точек.

    Parameters:
    -----------
    trail_length : int
        Длина следа за каждой точкой (в кадрах)
    """
    print(f"Генерация траекторий денойзинга...")

    n_classes = pipeline.unk_cond_id
    n_total = n_samples_per_class * n_classes
    conditions = torch.arange(n_classes, device=device).repeat_interleave(n_samples_per_class)

    history = ddpm_sampling_with_history(pipeline, n_total, conditions, device,
                                         num_steps=num_denoising_steps)

    print(f"Интерполяция траекторий...")
    history_interp = interpolate_trajectory(history, interpolation_factor)
    num_frames = history_interp.shape[0]

    print(f"Создание анимации с траекториями ({num_frames} кадров)...")

    fig, ax = plt.subplots(figsize=figsize, facecolor='black')
    ax.set_facecolor('black')

    all_points = history_interp.reshape(-1, 2).numpy()
    x_min, x_max = all_points[:, 0].min() - 1, all_points[:, 0].max() + 1
    y_min, y_max = all_points[:, 1].min() - 1, all_points[:, 1].max() + 1

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2, color='white')
    ax.tick_params(colors='white')

    colors = plt.cm.Set3(np.linspace(0, 1, n_classes))

    # Scatter plots для точек
    scatters = []
    # Line plots для траекторий
    trails = []

    for class_idx in range(n_classes):
        scatter = ax.scatter([], [], c=[colors[class_idx]], s=50, alpha=0.9,
                           edgecolors='white', linewidths=0.5,
                           label=f'Class {class_idx}', zorder=3)
        scatters.append(scatter)

        # Для каждой точки класса создаем линию траектории
        mask = (conditions.cpu() == class_idx).numpy()
        n_points_class = mask.sum()
        class_trails = []
        for _ in range(n_points_class):
            line, = ax.plot([], [], c=colors[class_idx], alpha=0.3, linewidth=1, zorder=1)
            class_trails.append(line)
        trails.append(class_trails)

    legend = ax.legend(facecolor='black', edgecolor='white',
                      loc='upper right', framealpha=0.8)
    for text in legend.get_texts():
        text.set_color('white')

    progress_text = ax.text(0.02, 0.98, '', transform=ax.transAxes,
                           color='white', fontsize=12, verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

    def init():
        for scatter in scatters:
            scatter.set_offsets(np.empty((0, 2)))
        for class_trails in trails:
            for line in class_trails:
                line.set_data([], [])
        progress_text.set_text('')
        return scatters + [item for sublist in trails for item in sublist] + [progress_text]

    def animate(frame):
        points = history_interp[frame].numpy()

        # Обновляем точки и траектории для каждого класса
        for class_idx, (scatter, class_trails) in enumerate(zip(scatters, trails)):
            mask = (conditions.cpu() == class_idx).numpy()
            class_points = points[mask]
            scatter.set_offsets(class_points)

            # Обновляем траектории
            class_indices = np.where(mask)[0]
            for local_idx, global_idx in enumerate(class_indices):
                # Траектория: от (frame - trail_length) до frame
                start_frame = max(0, frame - trail_length)
                trail_points = history_interp[start_frame:frame+1, global_idx].numpy()

                if len(trail_points) > 1:
                    class_trails[local_idx].set_data(trail_points[:, 0], trail_points[:, 1])
                else:
                    class_trails[local_idx].set_data([], [])

        progress = frame / (num_frames - 1) * 100
        timestep = int((1 - frame / (num_frames - 1)) * (num_denoising_steps - 1))
        progress_text.set_text(f'Progress: {progress:.1f}%\nTimestep: {timestep}')

        return scatters + [item for sublist in trails for item in sublist] + [progress_text]

    anim = FuncAnimation(fig, animate, init_func=init, frames=num_frames,
                        interval=1000/fps, blit=True, repeat=True)

    # Определяем формат по расширению файла
    file_ext = os.path.splitext(filename)[1].lower()

    print(f"Сохранение анимации в {filename}...")

    if file_ext in ['.mp4', '.avi', '.mov']:
        # Видео форматы
        try:
            writer = FFMpegWriter(fps=fps, bitrate=2000, codec='libx264')
            anim.save(filename, writer=writer, dpi=dpi)
            print(f"✅ Видео с траекториями сохранено: {filename}")
        except Exception as e:
            print(f"⚠️ Ошибка сохранения видео: {e}")
            print("Попробуйте установить ffmpeg: conda install ffmpeg или pip install ffmpeg-python")
            # Fallback на GIF
            fallback_name = filename.rsplit('.', 1)[0] + '.gif'
            print(f"Сохранение в GIF: {fallback_name}")
            writer = PillowWriter(fps=fps)
            anim.save(fallback_name, writer=writer, dpi=dpi)
            print(f"✅ GIF с траекториями сохранен: {fallback_name}")
    else:
        # GIF формат (по умолчанию)
        writer = PillowWriter(fps=fps)
        anim.save(filename, writer=writer, dpi=dpi)
        print(f"✅ GIF с траекториями сохранен: {filename}")

    plt.close(fig)

    return anim


In [ ]:
# Создаем анимацию с траекториями
# create_denoising_animation_with_trails(
#     pipeline=pipeline,
#     n_samples_per_class=30,      # Меньше точек для лучшей видимости траекторий
#     device=device,
#     num_denoising_steps=50,
#     interpolation_factor=5,
#     fps=30,
#     filename='denoising_with_trails.mp4',  # MP4 или GIF
#     figsize=(10, 10),
#     dpi=100,
#     trail_length=30              # Длина следа (в кадрах)
# )


In [ ]:
create_denoising_animation_with_trails(
    pipeline=pipeline,
    n_samples_per_class=20,      # Меньше точек для лучшей видимости траекторий
    device='cpu',
    num_denoising_steps=500,
    interpolation_factor=3,
    fps=60,
    filename='500_denoising_with_trails60fps.mp4',  # MP4 или GIF
    figsize=(10, 10),
    dpi=100,
    trail_length=800              # Длина следа (в кадрах)
)


## Описание анимационных функций

### `create_denoising_animation()`
Основная функция для создания анимации процесса денойзинга.

**Особенности:**
- 🎨 Черный фон
- 🌈 Цветовая кодировка по классам
- 📊 Отображение прогресса и текущего timestep
- 🎬 Кубическая интерполяция для плавности
- 💾 Сохранение в MP4, GIF, AVI, или MOV

**Форматы файлов:**
- **MP4** (рекомендуется) - отличное качество, маленький размер
- **GIF** - широкая поддержка, больший размер
- **AVI** - высокое качество, большой размер
- **MOV** - Apple формат

**Параметры:**
- `filename`: имя файла с расширением (.mp4, .gif, .avi, .mov)
- `n_samples_per_class`: количество точек на класс (50 = баланс скорость/качество)
- `num_denoising_steps`: шагов денойзинга (50 = хорошо, 100 = детально, 25 = быстро)
- `interpolation_factor`: множитель кадров (5 = плавно, 10 = очень плавно, 3 = быстро)
- `fps`: кадров в секунду (30 = стандарт, 60 = плавно, 15 = медленно)
- `dpi`: разрешение (100 = норм, 150 = HD, 80 = быстро)

### `create_denoising_animation_with_trails()`
Анимация с траекториями движения точек.

**Дополнительно:**
- 🌠 Следы за каждой точкой
- 🎯 Визуализация пути денойзинга
- `trail_length`: длина следа в кадрах (20-50 оптимально)

### Рекомендации по использованию:

**Для быстрого теста:**
```python
num_denoising_steps=25, interpolation_factor=3, fps=20, dpi=80
```

**Для презентации:**
```python
num_denoising_steps=50, interpolation_factor=5, fps=30, dpi=100
```

**Для публикации:**
```python
num_denoising_steps=100, interpolation_factor=10, fps=60, dpi=150
```


### Установка ffmpeg для MP4

Если при сохранении в MP4 возникает ошибка, установите ffmpeg:

**Conda:**
```bash
conda install ffmpeg
```

**Pip:**
```bash
pip install ffmpeg-python
```

**Windows (альтернатива):**
1. Скачайте ffmpeg с https://ffmpeg.org/download.html
2. Добавьте в PATH

Если ffmpeg недоступен, функция автоматически сохранит в GIF формат.
